# Baseline Flood Prediction Model — Random Forest
> Updated model to use a Random Forest Regressor. Limits depth to prevent overfitting given the limited data (5 stations).

| | |
|---|---|
| **Author** | Clayton Sasaki |
| **Last modified** | January 13, 2026 |
| **Competition** | [CodaBench #10801](https://www.codabench.org/competitions/10801/) |

In [ ]:
!git clone https://github.com/iharp-institute/iHARP-ML-Challenge-2.git

fatal: destination path 'iHARP-ML-Challenge-2' already exists and is not an empty directory.


In [ ]:
# ============================================
# Install & Import Dependencies
# ============================================
# !pip install scipy pandas scikit-learn

import pandas as pd
import numpy as np
from scipy.io import loadmat
from datetime import datetime, timedelta
import pickle

# for model
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import confusion_matrix, accuracy_score, f1_score, matthews_corrcoef

# ============================================
# Helper Function: MATLAB datenum → datetime
# ============================================
def matlab2datetime(matlab_datenum):
    return datetime.fromordinal(int(matlab_datenum)) \
           + timedelta(days=matlab_datenum % 1) \
           - timedelta(days=366)

# ============================================
# Load .mat Datasets
# ============================================
data = loadmat('/content/iHARP-ML-Challenge-2/NEUSTG_19502020_12stations.mat')
threshold_data = loadmat('/content/iHARP-ML-Challenge-2/Seed_Coastal_Stations_Thresholds.mat')

lat = data['lattg'].flatten()
lon = data['lontg'].flatten()
sea_level = data['sltg']
threshold = threshold_data['thminor_stnd']
station_names = [s[0] for s in data['sname'].flatten()]
time = data['t'].flatten()
time_dt = np.array([matlab2datetime(t) for t in time])

# ============================================
# Select Target Stations
# ============================================
SELECTED_STATIONS = [
    'Annapolis', 'Atlantic_City', 'Charleston', 'Washington', 'Wilmington'
]

selected_idx = [station_names.index(st) for st in SELECTED_STATIONS]
selected_names = [station_names[i] for i in selected_idx]
selected_lat = lat[selected_idx]
selected_lon = lon[selected_idx]
selected_threshold = threshold[selected_idx]
selected_sea_level = sea_level[:, selected_idx]

# ============================================
# Convert Hourly → Daily per Station
# ============================================
time_dt = pd.to_datetime(time_dt)

df_hourly = pd.DataFrame({
    'time': np.tile(time_dt, len(selected_names)),
    'station_name': np.repeat(selected_names, len(time_dt)),
    'latitude': np.repeat(selected_lat, len(time_dt)),
    'longitude': np.repeat(selected_lon, len(time_dt)),
    'sea_level': selected_sea_level.flatten(),
    'flood_threshold': np.repeat(selected_threshold, len(time_dt)),
})

# ============================================
# Daily Aggregation + Flood Flag
# ============================================
df_daily = df_hourly.groupby(['station_name', pd.Grouper(key='time', freq='D')]).agg({
    'sea_level': 'mean',
    'latitude': 'first',
    'longitude': 'first',
    'flood_threshold': 'first'
}).reset_index()

# Flood flag: 1 if any hourly value exceeded threshold that day
hourly_max = df_hourly.groupby(['station_name', pd.Grouper(key='time', freq='D')])['sea_level'].max().reset_index()
df_daily = df_daily.merge(hourly_max, on=['station_name','time'], suffixes=('','_max'))
df_daily['flood'] = (df_daily['sea_level_max'] > df_daily['flood_threshold']).astype(int)

# ============================================
# Feature Engineering (3d & 7d means)
# ============================================
df_daily['sea_level_3d_mean'] = df_daily.groupby('station_name')['sea_level'].transform(
    lambda x: x.rolling(3, min_periods=1).mean())
df_daily['sea_level_7d_mean'] = df_daily.groupby('station_name')['sea_level'].transform(
    lambda x: x.rolling(7, min_periods=1).mean())

In [ ]:
# ============================================
# Build 7-day → 14-day Training Windows
# ============================================
FEATURES = ['sea_level', 'sea_level_3d_mean', 'sea_level_7d_mean']
HIST_DAYS = 7
FUTURE_DAYS = 14

X_train, y_train = [], []

for stn, grp in df_daily.groupby('station_name'):
    grp = grp.sort_values('time').reset_index(drop=True)
    for i in range(len(grp) - HIST_DAYS - FUTURE_DAYS):
        hist = grp.loc[i:i+HIST_DAYS-1, FEATURES].values.flatten()
        future = grp.loc[i+HIST_DAYS:i+HIST_DAYS+FUTURE_DAYS-1, 'flood'].values
        X_train.append(hist)
        y_train.append(future)

X_train = np.array(X_train)
y_train = np.array(y_train)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")

# ============================================
# Select Historical Window (Manual / Random)
# ============================================
# --- Option 1: RANDOM window ---
# np.random.seed(42)
# date_range = pd.date_range(start='1950-01-01', end='2020-12-15')
# hist_start = np.random.choice(date_range)
# hist_end = hist_start + pd.Timedelta(days=6)

# --- Option 2: MANUAL window ---
hist_start = pd.to_datetime('2013-07-21')
hist_end   = pd.to_datetime('2013-07-27')

# Forecast period
test_start = hist_end + pd.Timedelta(days=1)
test_end   = test_start + pd.Timedelta(days=13)

X_train shape: (129560, 21)
y_train shape: (129560, 14)


In [ ]:
# ============================================
# Build X_test for Selected Window
# ============================================
X_test = []
for stn, grp in df_daily.groupby('station_name'):
    mask = (grp['time'] >= hist_start) & (grp['time'] <= hist_end)
    hist_block = grp.loc[mask, FEATURES].values.flatten()
    if len(hist_block) == 7 * len(FEATURES):
        X_test.append(hist_block)

X_test = np.array(X_test)

# ============================================
# Train 14 RandomForest Models (1 per forecast day)
# ============================================
print("Training Random Forest models...")
models = []
for d in range(14):
    # --- Using RandomForestRegressor ---
    model = RandomForestRegressor(
        n_estimators=20,      # Number of trees
        max_depth=10,          # Limits depth to prevent overfitting
        n_jobs=-1,             # Use all CPU cores
        random_state=42
    )
    model.fit(X_train, y_train[:, d])
    models.append(model)

# Save the model
with open("rf_models.pkl", "wb") as f:
    pickle.dump(models, f)

Training Random Forest models...


In [ ]:
# ============================================
# Forecast 14 Days Ahead
# ============================================
y_pred = np.array([m.predict(X_test) for m in models]).T
y_pred_bin = (y_pred > 0.5).astype(int)

# ============================================
# Collect Ground Truth
# ============================================
y_true = []
for stn, grp in df_daily.groupby('station_name'):
    mask = (grp['time'] >= test_start) & (grp['time'] <= test_end)
    vals = grp.loc[mask, 'flood'].values
    if len(vals) == 14:
        y_true.append(vals)
y_true = np.array(y_true)

# ============================================
# Evaluation
# ============================================
y_true_flat = y_true.flatten()
y_pred_flat = y_pred_bin.flatten()

tn, fp, fn, tp = confusion_matrix(y_true_flat, y_pred_flat).ravel()
acc = accuracy_score(y_true_flat, y_pred_flat)
f1 = f1_score(y_true_flat, y_pred_flat)
mcc = matthews_corrcoef(y_true_flat, y_pred_flat)

print("=== Confusion Matrix ===")
print(f"TP: {tp} | FP: {fp} | TN: {tn} | FN: {fn}")
print("\n=== Metrics ===")
print(f"Accuracy: {acc:.3f}")
print(f"F1 Score: {f1:.3f}")
print(f"MCC: {mcc:.3f}")

=== Confusion Matrix ===
TP: 20 | FP: 1 | TN: 36 | FN: 13

=== Metrics ===
Accuracy: 0.800
F1 Score: 0.741
MCC: 0.631
